# Interactive Inference: Qwen2.5-VL + DisasterM3 LoRA
Use this notebook to quickly test your fine-tuned model on individual images.

In [ ]:
!pip install -q -U torch torchvision transformers peft accelerate bitsandbytes qwen-vl-utils

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

# 1. Load Base Model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading base model...")
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# 2. Load Your LoRA Adapter
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model, 
    "AbrarAlam/disasterm3-qwen25vl7b-qlora", # Ensure this is your HF repo
    subfolder="checkpoint-217"               # Update if your final checkpoint is different
)
print("Model ready for testing!")

In [ ]:
from qwen_vl_utils import process_vision_info

def ask_model(image_url_or_path, question):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_url_or_path},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    return processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

In [ ]:
# Test it!
image_url = "https://example.com/disaster_image.jpg" # Replace with a real image URL or path
question = "Identify the disaster type in the image and describe the structural damage."

# answer = ask_model(image_url, question)
# print("Model Answer:", answer)